In [2]:
import pandas as pd
df = pd.read_csv("clean_data.csv")
df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   OrderID          1200 non-null   object 
 1   Date             1200 non-null   object 
 2   CustomerID       1200 non-null   object 
 3   Product          1200 non-null   object 
 4   Quantity         1200 non-null   int64  
 5   UnitPrice        1200 non-null   float64
 6   ShippingAddress  1200 non-null   object 
 7   PaymentMethod    1200 non-null   object 
 8   OrderStatus      1200 non-null   object 
 9   TrackingNumber   1200 non-null   object 
 10  ItemsInCart      1200 non-null   int64  
 11  CouponCode       1200 non-null   object 
 12  ReferralSource   1200 non-null   object 
 13  TotalPrice       1200 non-null   float64
dtypes: float64(2), int64(2), object(10)
memory usage: 131.4+ KB


In [5]:
# 1. Total revenue across all orders
total_revenue = df['TotalPrice'].sum()
print(total_revenue)

1264761.96


In [6]:
# 2. Average price of items sold
avg_unit_price = df['UnitPrice'].mean()
print(avg_unit_price)

356.41275


In [7]:
# 3. Total number of transactions
total_orders = df['OrderID'].count()
print(total_orders)

1200


In [16]:
product_sales=df.groupby("Product")["TotalPrice"].sum()
print(product_sales)

Product
Chair      195620.11
Desk       167459.93
Laptop     192126.56
Monitor    175651.41
Phone      151722.39
Printer    195612.61
Tablet     186568.95
Name: TotalPrice, dtype: float64


In [21]:
# Custom business report per product category
product_report = df.groupby('Product').agg(
    Total_Revenue=('TotalPrice', 'sum'),
    Avg_Unit_Price=('UnitPrice', 'mean'),
    Total_Orders=('OrderID', 'count')
).reset_index()

print(product_report)

   Product  Total_Revenue  Avg_Unit_Price  Total_Orders
0    Chair      195620.11      355.660281           178
1     Desk      167459.93      329.608118           170
2   Laptop      192126.56      357.707052           173
3  Monitor      175651.41      358.663620           163
4    Phone      151722.39      375.217564           156
5  Printer      195612.61      351.714917           181
6   Tablet      186568.95      367.679106           179


In [ ]:
# Rows = Product, Columns = Payment Method, Values = Sum of TotalPrice
revenue_matrix = df.pivot_table(
    index='Product', 
    columns='PaymentMethod', 
    values='TotalPrice', 
    aggfunc='sum',
    fill_value=0
)

print(revenue_matrix)

PaymentMethod      Cash  Credit Card  Debit Card  Gift Card    Online
Product                                                              
Chair          35902.59     35405.05    31906.21   38064.35  54341.91
Desk           31644.97     37473.98    29316.48   39498.39  29526.11
Laptop         48676.40     35308.12    35713.68   42057.39  30370.97
Monitor        32847.01     42473.64    31790.11   33301.19  35239.46
Phone          25997.29     32317.43    31731.75   31525.70  30150.22
Printer        46940.27     43289.52    33165.09   32679.23  39538.50
Tablet         37777.76     37579.89    38737.86   29197.67  43275.77


In [23]:
# Take the 2D grid from Level 4 and flatten it back into a tidy tall table
tidy_data = revenue_matrix.reset_index().melt(
    id_vars='Product', 
    var_name='PaymentMethod', 
    value_name='TotalSales'
)

print(tidy_data.head())

   Product PaymentMethod  TotalSales
0    Chair          Cash    35902.59
1     Desk          Cash    31644.97
2   Laptop          Cash    48676.40
3  Monitor          Cash    32847.01
4    Phone          Cash    25997.29


In [24]:
df['Date'] = pd.to_datetime(df['Date'])

# 2. Set 'Date' as the index and sort chronologically
df_time = df.set_index('Date').sort_index()

# Now you can slice data by year or date range instantly!
sales_2023 = df_time.loc['2023']
q1_2024 = df_time.loc['2024-01-01':'2024-03-31']

In [25]:
# Aggregate daily sales into Monthly totals ('M')
monthly_sales = df_time['TotalPrice'].resample('M').sum()
print(monthly_sales.head(3))

Date
2023-01-31    56685.75
2023-02-28    40117.66
2023-03-31    48609.37
Freq: ME, Name: TotalPrice, dtype: float64


C:\Users\Computer World\AppData\Local\Temp\ipykernel_5264\1887585205.py:2: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_sales = df_time['TotalPrice'].resample('M').sum()


In [26]:
# Convert monthly series to DataFrame
monthly_df = monthly_sales.to_frame()

# Calculate a 3-Month Moving Average
monthly_df['3M_Moving_Avg'] = monthly_df['TotalPrice'].rolling(window=3).mean()
print(monthly_df.head(5))

            TotalPrice  3M_Moving_Avg
Date                                 
2023-01-31    56685.75            NaN
2023-02-28    40117.66            NaN
2023-03-31    48609.37   48470.926667
2023-04-30    27751.71   38826.246667
2023-05-31    63836.84   46732.640000


In [27]:
# Calculate Month-over-Month Growth Percentage
monthly_df['MoM_Growth_%'] = monthly_df['TotalPrice'].pct_change() * 100
print(monthly_df.head(4))

            TotalPrice  3M_Moving_Avg  MoM_Growth_%
Date                                               
2023-01-31    56685.75            NaN           NaN
2023-02-28    40117.66            NaN    -29.227963
2023-03-31    48609.37   48470.926667     21.167012
2023-04-30    27751.71   38826.246667    -42.908723


In [28]:
import numpy as np

df['Tier'] = np.where(df['TotalPrice'] >= 1000, 'High Value', 'Standard Value')

In [29]:
# Define conditions list
conditions = [
    df['TotalPrice'] >= 1500,  # Tier 1
    df['TotalPrice'] >= 500    # Tier 2
]

# Define corresponding outputs
choices = ['High Value', 'Medium Value']

# Execute vectorized lookup (default = 'Low Value')
df['OrderTier'] = np.select(conditions, choices, default='Low Value')

print(df[['TotalPrice', 'OrderTier']].head(4))

   TotalPrice   OrderTier
0     2853.10  High Value
1      302.70   Low Value
2     2753.40  High Value
3      273.19   Low Value


In [30]:
# Check memory usage before
mem_before = df['PaymentMethod'].memory_usage(deep=True)

# Convert string object column to categorical type
df['PaymentMethod'] = df['PaymentMethod'].astype('category')

# Check memory usage after
mem_after = df['PaymentMethod'].memory_usage(deep=True)

print(f"Memory Before: {mem_before} bytes")
print(f"Memory After:  {mem_after} bytes")

Memory Before: 78024 bytes
Memory After:  1825 bytes
